# Structured Multimodal RAG: typed rows, visual regions, and cited evidence

## NovaTech renewal-risk investigation

A customer-success lead asks which accounts require attention. The evidence lives in a typed renewal table and a dashboard screenshot/OCR extraction. We calculate only with typed values, treat OCR as uncertain visual observation, and retain separate citations.


## Evidence-routing map

```text
Question
  |-- numeric filter / aggregation --> typed table or SQL --> row citations
  |-- visual label / chart / scan ----> OCR or vision parser -> region citations
  |-- policy / narrative -------------> text retrieval ------> passage citations
  +-- combined recommendation --------> evidence bundle -----> verified answer
```

The key design choice is the operation: an LLM must not replace a governed calculation just because the source started as text.


In [ ]:
from src.rag_core.lesson_loader import load_lesson_module; globals().update({name: value for name, value in vars(load_lesson_module('curriculum/advanced/04-structured-multimodal/lab.py')).items() if not name.startswith('_')})

rows = [
    TableRow("acme-17", {"account": "Acme", "risk_usd": 125000, "currency": "USD", "as_of": "2026-08-01"}, "renewals.csv#17"),
    TableRow("globex-03", {"account": "Globex", "risk_usd": 40000, "currency": "USD", "as_of": "2026-08-01"}, "renewals.csv#03"),
]
assert validate_table_rows(rows, {"account", "risk_usd", "currency", "as_of"}) == []


## 1 — Deterministic table calculation

Filter and aggregate in code/SQL, not in model prose. Keep the list of rows in the calculation result so a response can show exactly which records contributed.


In [ ]:
acme = filter_rows(rows, account="Acme")
summary, row_citations = aggregate_with_citations(acme, "risk_usd")
print(summary)
print(row_citations)
assert summary["sum"] == 125000
assert row_citations[0].locator == "row=acme-17"


## 2 — Schema drift and unit safety

A valid-looking row can still be unsafe: missing currency, a date in the wrong timezone, or a percentage mixed with dollars. Fail closed before aggregation, then ask for corrected data or an approved conversion policy.


In [ ]:
bad_rows = rows + [TableRow("acme-18", {"account": "Acme", "risk_usd": 90000}, "renewals.csv#18")]
errors = validate_table_rows(bad_rows, {"account", "risk_usd", "currency", "as_of"})
print(errors)
assert "missing as_of, currency" in errors[0]
assert {row.values.get("currency") for row in rows} == {"USD"}


## 3 — OCR is located, uncertain evidence

An OCR string without page and bounding box cannot be reviewed. Low-confidence regions should not enter automatic answer context. This fixture has one reliable and one unreliable reading of the same dashboard label.


In [ ]:
regions = [
    OcrRegion("dashboard-warning", "renewal-dashboard", 1, (80, 290, 760, 45), "Validate Acme migration before renewal", 0.98, "dashboard.svg"),
    OcrRegion("dashboard-noise", "renewal-dashboard", 1, (80, 200, 260, 55), "Acme migration", 0.42, "dashboard.svg"),
]
hits = search_ocr_regions("Validate Acme migration", regions, min_confidence=0.8)
visual_citations = citations_for_regions(hits)
print(hits)
print(visual_citations)
assert [hit.region_id for hit in hits] == ["dashboard-warning"]


## 4 — Compose, but do not conflate, modalities

A safe brief separates computed facts, observed visual evidence, and recommendation. It never states that an OCR observation *caused* a numeric risk unless another source establishes causality.


In [ ]:
bundle = {
    "computed": {"claim": "Acme has $125,000 in listed renewal risk.", "citations": row_citations},
    "observed": {"claim": "The dashboard contains a migration-validation warning for Acme.", "citations": visual_citations},
    "inference": "Escalate Acme for human review; the sources do not prove the warning is the cause of the risk.",
}
print(bundle)
assert citations_are_known(row_citations + visual_citations, row_ids={"acme-17"}, region_ids={"dashboard-warning"})


## 5 — Route by operation and risk

A query router should select a deterministic data path before a model writes prose. This lightweight example is intentionally transparent. In production, use a typed classifier/contract and evaluate its routing accuracy on real requests.


In [ ]:
def route_question(question: str) -> str:
    q = question.lower()
    if any(word in q for word in ("total", "average", "sum", "how many")):
        return "structured-query"
    if any(word in q for word in ("dashboard", "chart", "screenshot", "image")):
        return "vision-or-ocr"
    if any(word in q for word in ("policy", "requirement", "procedure")):
        return "text-retrieval"
    return "human-triage-or-hybrid"

for q in ["What is Acme's total risk?", "What does the dashboard warning say?", "What does the renewal policy require?", "Should we contact Acme?"]:
    print(q, "=>", route_question(q))
assert route_question("What is Acme's total risk?") == "structured-query"


## 6 — Currency, units, and time are part of the schema

A numeric column is not enough. Before any calculation, require compatible currency, unit, as-of date, and conversion authority. Never allow a model to silently convert or combine incompatible values.


In [ ]:
def safe_total(rows, column: str, currency: str, as_of: str) -> int | float:
    selected = [row for row in rows if row.values.get("currency") == currency and row.values.get("as_of") == as_of]
    if len(selected) != len(rows):
        raise ValueError("mixed currency or as-of date requires an approved conversion/reconciliation route")
    return sum(row.values[column] for row in selected)

print(safe_total(rows, "risk_usd", "USD", "2026-08-01"))
mixed = rows + [TableRow("acme-eur", {"account": "Acme", "risk_usd": 80000, "currency": "EUR", "as_of": "2026-08-01"}, "renewals.csv#19")]
try:
    safe_total(mixed, "risk_usd", "USD", "2026-08-01")
except ValueError as error:
    print("Blocked:", error)


## 7 — Tenant isolation is enforced before evidence selection

Two tenants can have the same account name. A model must not see the other tenant’s row, asset title, OCR text, or citation. This simplified filter represents a database view or policy-aware retrieval API in production.


In [ ]:
tenant_rows = [
    TableRow("northstar-acme", {"tenant": "northstar", "account": "Acme", "risk_usd": 125000}, "northstar.csv#17"),
    TableRow("other-acme", {"tenant": "other", "account": "Acme", "risk_usd": 900000}, "other.csv#17"),
]
def rows_for_tenant(rows, tenant):
    return [row for row in rows if row.values.get("tenant") == tenant]
visible = rows_for_tenant(tenant_rows, "northstar")
print(visible)
assert [row.row_id for row in visible] == ["northstar-acme"]


## 8 — Verify claims against the correct modality

A claim checker should reject a numeric claim with only an OCR citation, or a dashboard observation with only a table row. This catches a subtle but important failure: citations that exist but do not actually support the claim type.


In [ ]:
def claim_is_supported(claim_type: str, citations) -> bool:
    required = {"numeric": "table", "visual": "ocr", "policy": "text"}[claim_type]
    return any(citation.modality == required for citation in citations)

assert claim_is_supported("numeric", row_citations)
assert claim_is_supported("visual", visual_citations)
assert not claim_is_supported("numeric", visual_citations)
print("Numeric and visual claims have different evidence requirements.")


## 9 — Low-confidence visual evidence escalates

The correct response to unreadable input is not guessing. Keep the rejected region and reason in the trace, then ask for a higher-quality source or human review.


In [ ]:
low_confidence = search_ocr_regions("Acme migration", regions, min_confidence=0.99)
if not low_confidence:
    review = {"status": "needs-human-review", "reason": "no OCR region met the confidence threshold", "asset": "renewal-dashboard"}
    print(review)
assert not low_confidence


## 10 — Evaluate an end-to-end multimodal answer

A production evaluation record scores each layer, then uses a release gate. Notice that a fluent answer is not sufficient: the route, arithmetic, citations, and confidence handling must all pass.


In [ ]:
evaluation = {
    "route_correct": route_question("What is Acme's total risk?") == "structured-query",
    "numeric_correct": summary["sum"] == 125000,
    "row_citations_valid": citations_are_known(row_citations, row_ids={"acme-17"}, region_ids=set()),
    "visual_citations_valid": citations_are_known(visual_citations, row_ids=set(), region_ids={"dashboard-warning"}),
    "low_confidence_blocked": not low_confidence,
    "tenant_isolated": [row.row_id for row in visible] == ["northstar-acme"],
}
evaluation["releaseable"] = all(evaluation.values())
print(evaluation)
assert evaluation["releaseable"]


## 5 — Production evaluation

Score numeric correctness, unit/currency correctness, schema-valid rate, row-level/region-level citation correctness, OCR confidence calibration, modality-route accuracy, tenant isolation, and answer claim support. Test scanned PDFs, merged tables, rotated pages, chart legends, low-quality images, schema drift, and injected text in documents.


## Exercises

1. Add a EUR row and require a dated, cited conversion rate before aggregation.
2. Add a second tenant’s Acme account and enforce row/asset filtering.
3. Add an unlabeled chart axis; identify what is safe to report and what requires review.
4. Add a table claim verifier that rejects numeric prose without row citations.
5. Compare text-only RAG with this modality-aware route over ten renewal-risk questions.

Read the companion [lesson](README.md) for architecture, technology choices, and production checklist.
